# 08 — Persona Santander: chân dung và hành động

Notebook mới, nối output 01 (RFM features), 05 (lifecycle), tùy chọn output 07 (CLV). Chưa chạy trên dữ liệu Santander trong phiên này. Đã đọc source notebook 03/05/06/07 ngày 15/10/2026; không sao chép kết quả cũ thành kết quả persona.

**Điều chỉnh theo pipeline thực tế:** M_group_points là propensity score upstream, không phải tiền hay số sản phẩm. Độ rộng danh mục lấy từ portfolio_code riêng. Lifecycle New dựa ngày mở quan hệ; At-risk/Churned là vắng snapshot; Win-back là quay lại panel. Không gọi các trạng thái này là churn ngân hàng đã xác nhận.

Persona là lớp diễn giải có rule, không phải một mô hình clustering mới. Rule v1 là giả thuyết nghiệp vụ, cần kiểm chứng. Không tuyên bố F4 tốt nhất chỉ vì dùng F4.

## Chạy
Thêm output notebook 01 và 05 vào Kaggle. Điền FEATURE_ROOT/LIFECYCLE_ROOT nếu tự tìm thấy nhiều bản. Run All. CLV_PATH tùy chọn là file dự báo **1 tháng** của notebook 07 tại RECOMMEND_MONTH. Cần numpy, pandas, pyarrow, matplotlib, nbformat (chỉ dùng khi kiểm tra notebook ngoài Kaggle).

Nguồn tham chiếu: 03-rfm-quantile-333-555-1(3).ipynb; 05-lifecycle-eda-insights(1).ipynb; 06-affinity-rfm-lifecycle.ipynb; 07-clv-proxy (4).ipynb.

## Bổ sung v2 — lịch sử event và cửa sổ 3/6 tháng
Tách **NoObservedEvent** khỏi khách đã từng có event. Với khách HasEvent, chia: có event trong 3 tháng; không event 3 tháng nhưng có trong 6; không event 6 tháng; không event 3 tháng nhưng chưa xác nhận đủ 6; chưa đủ quan sát để kết luận 3 tháng.

Event cửa sổ lấy trực tiếp từ `acquisition_pair_valid` và `has_acquisition_month` của notebook 05, không suy ra không-event từ snapshot thiếu. Cửa sổ 3 tháng tại t gồm event tại t-2, t-1, t: cần snapshot từ t-3 đến t để có đủ 3 cặp. Tương tự cửa sổ 6 tháng cần 7 snapshot liên tiếp đầy đủ. Một event hợp lệ đã quan sát đủ để xác nhận “có”; xác nhận “không” cần mọi cặp trong cửa sổ hợp lệ.

`event_history_group` là lớp riêng; `persona_base` giữ rule v1. Persona mới kết hợp lớp event với số sản phẩm (0–2 / 3 / ≥4), đồng thời ưu tiên trạng thái lifecycle/chất lượng dữ liệu như trước. F4 giữ nguyên để mô tả; nhóm recent tiếp tục phân biệt F>=3 và F<3. Không khẳng định cách chia này tối ưu trước khi chạy lại.

**Các output đã xóa vì rule thay đổi. Cần Run All để có kết quả v2.**


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
FEATURE_ROOT=None
LIFECYCLE_ROOT=None
CLV_PATH=None
RECOMMEND_MONTH='2016-05'
EVAL_MONTHS=['2015-12','2016-01','2016-02','2016-03','2016-04']
OUT=Path('/kaggle/working/persona_outputs' if Path('/kaggle/working').exists() else './persona_outputs')
OUT.mkdir(parents=True,exist_ok=True)
RULE_VERSION='persona-v2-event-windows-F4'
MIN_VALID_PAIRS=3
RECENT_R=2
FREQUENT_F=3  # F4 level 3–4: F >= 3; không phải chỉ riêng level 4
WIDE_PRODUCTS=4
NARROW_PRODUCTS=2
MIN_GROUP_N=1000
MIN_POSITIVES=20
# Ngưỡng nghiệp vụ ứng viên, khóa trước đọc bảng đánh giá; không học lại theo tháng.
SEARCH_ROOT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
def resolve(root, pattern):
    candidates=list((Path(root) if root else SEARCH_ROOT).rglob(pattern))
    if root and (Path(root)/pattern).exists():candidates=[Path(root)/pattern]
    parents=sorted(set(x.parent for x in candidates))
    if len(parents)!=1:raise ValueError(f'{pattern}: cần chọn folder duy nhất, có {parents}')
    return parents[0]
def inventory(root,prefix):
    return {x.stem.removeprefix(prefix):x for x in root.glob(prefix+'????-??.parquet')}


## 1. Rule có thứ tự ưu tiên

| Ưu tiên | Persona | Điều kiện |
|---|---|---|
| 1 | Vắng snapshot ngắn / dài | is_observed=False; giữ đúng At-risk/Churned upstream |
| 2 | Chưa đủ thông tin | Snapshot sản phẩm thiếu/không hợp lệ |
| 3 | Quan hệ mới | Lifecycle New đã xác minh upstream; không suy ra từ lần đầu thấy |
| 4 | Vừa quay lại panel | Win-back; không diễn giải thành acquisition |
| 5 | Lịch sử quan sát ngắn | Ít hơn 3 cặp tháng liên tiếp đủ sản phẩm |
| 6 | Thiếu lịch sử acquisition | F/R thiếu hoặc mâu thuẫn |
| 7 | Đang mở rộng lặp lại | R ≤ 2 và F_event_months ≥ 3 |
| 8 | Danh mục rộng, ít mở gần đây | ≥4 sản phẩm và không có event quan sát được hoặc R > 2 |
| 9 | Danh mục hẹp, ít mở gần đây | ≤2 sản phẩm và không có event quan sát được hoặc R > 2 |
| 10 | Danh mục khác | Phần còn lại; theo dõi trước khi tách thêm |

F4: **0 = NoObservedEvent riêng**, 1 → F1; 2 → F2; 3–4 → F3; ≥5 → F4. M giữ để mô tả, không dùng M thay số sản phẩm. Khách không có event không gán R=0. “Ít mở gần đây” chỉ nói về acquisition, không nói về giao dịch sử dụng ngân hàng.

**V2:** bảng rule trên tạo `persona_base`. Lớp event bổ sung bên dưới tạo `persona` cuối. Nhóm F=0 vẫn là “chưa quan sát event”, không có nghĩa chưa bao giờ mở sản phẩm trong đời. `no_event_3m`/`no_event_6m` có thể chưa xác định kể cả khi F=0 nếu thiếu coverage cửa sổ.


In [ ]:
def f4_label(f):
    return pd.Series(np.select([f.eq(0),f.eq(1),f.eq(2),f.between(3,4),f.ge(5)],
        ['NoObservedEvent','F1','F2','F3','F4'],default='Unknown'),index=f.index)

def assign_persona(d):
    d=d.copy()
    required=['customer_id','is_observed','lifecycle','products_complete','R','F_event_months','n_products','valid_pairs']
    missing=set(required)-set(d)
    if missing:raise ValueError(f'Thiếu cột: {missing}')
    assert d.customer_id.notna().all() and d.customer_id.is_unique
    assert d.is_observed.isin([True,False]).all()
    obs=d.is_observed.eq(True); complete=d.products_complete.eq(True)&d.n_products.notna()
    f=pd.to_numeric(d.F_event_months,errors='coerce');r=pd.to_numeric(d.R,errors='coerce')
    valid_f=f.notna()&f.ge(0)&f.mod(1).eq(0)
    good_history=valid_f & (f.eq(0) | (r.notna()&r.ge(0)))
    quiet=f.eq(0)|(f.gt(0)&r.gt(RECENT_R))
    conditions=[~obs&d.lifecycle.eq('Churned'),~obs,~complete,
        d.lifecycle.eq('New'),d.lifecycle.eq('Win-back'),d.valid_pairs.lt(MIN_VALID_PAIRS),
        ~good_history,r.le(RECENT_R)&f.ge(FREQUENT_F),
        quiet&d.n_products.ge(WIDE_PRODUCTS),quiet&d.n_products.le(NARROW_PRODUCTS)]
    labels=['Vắng snapshot dài','Vắng snapshot ngắn','Chưa đủ thông tin',
        'Quan hệ mới','Vừa quay lại panel','Lịch sử quan sát ngắn','Thiếu lịch sử acquisition',
        'Đang mở rộng lặp lại','Danh mục rộng, ít mở gần đây','Danh mục hẹp, ít mở gần đây']
    d['persona']=np.select([x.fillna(False).to_numpy(bool) for x in conditions],labels,default='Danh mục khác')
    d['F4_level']=f4_label(f).where(obs,'NoCurrentSnapshot')
    d['rule_version']=RULE_VERSION
    return d


def attach_event_history(d, recent, month):
    d=d.copy();ids=pd.Index(d.customer_id)
    for h in (3,6):
        wanted=[str(pd.Period(month,'M')-j) for j in range(h)]
        valid=np.zeros(len(d),int);positive=np.zeros(len(d),int)
        for mm in wanted:
            if mm not in recent:continue
            z=recent[mm].reindex(ids)
            valid+=z.valid.fillna(False).to_numpy(bool)
            positive+=z.positive.fillna(False).to_numpy(bool)
        d[f'valid_pairs_{h}m']=valid
        d[f'event_months_{h}m']=positive
        # True: đủ mọi cặp và không event; False: đã có ít nhất 1 event; NA: chưa kết luận.
        flag=pd.Series(pd.NA,index=d.index,dtype='boolean')
        flag.loc[valid==h]=True
        flag.loc[positive>0]=False
        flag.loc[~d.is_observed.eq(True)]=pd.NA
        d[f'no_event_{h}m']=flag
        d[f'window_{h}m_status']=np.where(positive>0,'ObservedEvent',np.where(valid==h,'ConfirmedNoEvent','InsufficientObservation'))
        d.loc[~d.is_observed.eq(True),f'window_{h}m_status']='NoCurrentSnapshot'
    f=pd.to_numeric(d.F_event_months,errors='coerce')
    valid_f=f.notna()&np.isfinite(f)&f.ge(0)&f.mod(1).eq(0)
    any3=d.event_months_3m.gt(0);any6=d.event_months_6m.gt(0)
    full3=d.valid_pairs_3m.eq(3);full6=d.valid_pairs_6m.eq(6)
    assert not (d.is_observed.eq(True)&f.eq(0)&any6).any(), 'F=0 nhưng có event hợp lệ gần đây: kiểm tra đồng bộ nguồn 01/05'
    d['event_history_group']=np.select([
        ~d.is_observed.eq(True),~valid_f,f.eq(0),any3,
        full3&any6,full6&~any6,full3&~any3],
        ['NoCurrentSnapshot','UnknownEventHistory','NoObservedEvent',
         'RecentEvent3m','NoEvent3m_EventWithin6m','NoEvent6m',
         'NoEvent3m_Unknown6m'],default='InsufficientRecentObservation')
    d['ever_observed_event']=pd.array(np.where(valid_f,f.gt(0),pd.NA),dtype='boolean')
    d.loc[~d.is_observed.eq(True),'ever_observed_event']=pd.NA
    d['persona_base']=d.persona
    # Lifecycle và chất lượng dữ liệu giữ ưu tiên; không biến Win-back thành acquisition.
    protected=d.persona_base.isin(['Vắng snapshot dài','Vắng snapshot ngắn',
        'Chưa đủ thông tin','Quan hệ mới','Vừa quay lại panel',
        'Lịch sử quan sát ngắn','Thiếu lịch sử acquisition'])
    descriptions={
        'NoObservedEvent':'Chưa quan sát event',
        'NoEvent3m_EventWithin6m':'Không event 3 tháng, có trong 6 tháng',
        'NoEvent6m':'Không event 6 tháng',
        'NoEvent3m_Unknown6m':'Không event 3 tháng, thiếu cửa sổ 6 tháng',
        'InsufficientRecentObservation':'Chưa đủ quan sát 3 tháng',
        'UnknownEventHistory':'Chưa rõ lịch sử event'}
    prefix=d.event_history_group.map(descriptions).fillna('Chưa rõ lịch sử event')
    recent_group=d.event_history_group.eq('RecentEvent3m')
    prefix=prefix.mask(recent_group&f.ge(FREQUENT_F),'Có event 3 tháng, mở lặp lại')
    prefix=prefix.mask(recent_group&f.lt(FREQUENT_F),'Có event 3 tháng, tần suất thấp')
    breadth=pd.Series(np.select([d.n_products.le(NARROW_PRODUCTS),d.n_products.ge(WIDE_PRODUCTS)],
        ['danh mục hẹp','danh mục rộng'],default='danh mục 3 sản phẩm'),index=d.index)
    d['portfolio_breadth']=breadth.where(d.products_complete.eq(True),'Unknown')
    d.loc[~protected,'persona']=(prefix+' | '+breadth).loc[~protected]
    return d


## 2. Nạp theo tháng và đếm exposure đúng
Chỉ đếm cặp snapshot liên tiếp đủ 24 sản phẩm; không dùng thời gian từ first_seen làm số tháng quan sát hợp lệ. Không mang portfolio cũ sang tháng vắng. Không nối toàn bộ customer-month vào RAM. portfolio_code phải theo cùng 24 sản phẩm upstream.

In [ ]:
feature_dir=resolve(FEATURE_ROOT,'rfm_????-??.parquet')
life_dir=resolve(LIFECYCLE_ROOT,'lifecycle_????-??.parquet')
feature_files=inventory(feature_dir,'rfm_');life_files=inventory(life_dir,'lifecycle_')
months=sorted(m for m in feature_files if m<=RECOMMEND_MONTH)
assert months and RECOMMEND_MONTH in months
assert all(str(pd.Period(a,'M')+1)==b for a,b in zip(months,months[1:]))
assert set(months)<=set(life_files),'Thiếu lifecycle tháng'
def load_features(m):
    q=pd.read_parquet(feature_files[m])
    required=['customer_id','products_complete','portfolio_code']
    assert set(required)<=set(q),f'{m}: cần {required}'
    assert q.customer_id.notna().all() and q.customer_id.is_unique
    c=pd.to_numeric(q.portfolio_code,errors='coerce')
    valid=c.notna()&c.ge(0)&c.lt(2**24)&c.mod(1).eq(0)&q.products_complete.eq(True)
    q['products_complete']=valid
    bits=c.where(valid,0).to_numpy(dtype=np.uint32)
    count=np.zeros(len(q),dtype=np.int16)
    for k in range(24):count+=((bits>>k)&1).astype(np.int16)
    q['n_products']=np.where(valid,count,np.nan)
    return q
history=pd.Series(dtype='int64');previous_complete=pd.Index([]);previous=None
profile_rows=[];evaluation_rows=[];transition_rows=[];recent_events={};event_profile_rows=[]
for m in months:
    q=load_features(m);meta=pd.read_parquet(life_files[m])
    assert meta.customer_id.is_unique and meta.customer_id.notna().all()
    assert meta.month_idx.eq(pd.Period(m,'M').year*12+pd.Period(m,'M').month).all()
    assert set(q.customer_id)==set(meta.loc[meta.is_observed.eq(True),'customer_id']), 'Nguồn lifecycle/features khác cohort'
    current_complete=pd.Index(q.loc[q.products_complete,'customer_id'])
    pairs=current_complete.intersection(previous_complete)
    history=history.reindex(history.index.union(q.customer_id),fill_value=0)
    history.loc[pairs]+=1
    # Features hiện tại là nguồn R/F/M/danh mục; metadata là nguồn lifecycle.
    cols=[c for c in ['customer_id','R','F_event_months','M_group_points','products_complete','n_products'] if c in q]
    base=meta.drop(columns=[c for c in cols if c!='customer_id' and c in meta])
    d=base.merge(q[cols],on='customer_id',how='left',validate='one_to_one')
    d['valid_pairs']=d.customer_id.map(history).fillna(0)
    d=assign_persona(d);d['as_of_month']=m
    assert meta.acquisition_pair_valid.isin([True,False]).all()
    assert not (meta.acquisition_pair_valid.eq(True)&meta.has_acquisition_month.isna()).any(), 'Cặp hợp lệ thiếu nhãn event'
    z=meta.set_index('customer_id')
    recent_events[m]=pd.DataFrame({'valid':z.acquisition_pair_valid.eq(True),
        'positive':z.acquisition_pair_valid.eq(True)&z.has_acquisition_month.eq(True).fillna(False)})
    recent_events={mm:zz for mm,zz in recent_events.items() if pd.Period(mm,'M')>=pd.Period(m,'M')-5}
    d=attach_event_history(d,recent_events,m)
    for label,g in d.groupby('event_history_group'):
        event_profile_rows.append(dict(as_of_month=m,event_history_group=label,n=len(g),share=len(g)/len(d),
            valid_pairs_3m_mean=g.valid_pairs_3m.mean(),valid_pairs_6m_mean=g.valid_pairs_6m.mean()))
    for label,g in d.groupby('persona'):
        row=dict(as_of_month=m,persona=label,n=len(g),share=len(g)/len(d),
            R_median=g.R.median(),F_median=g.F_event_months.median(),
            products_median=g.n_products.median(),valid_pairs_median=g.valid_pairs.median())
        if 'M_group_points' in g:row['M_points_median']=g.M_group_points.median()
        profile_rows.append(row)
    d.to_parquet(OUT/f'personas_{m}.parquet',index=False)
    if previous is not None:
        source_month=str(pd.Period(m,'M')-1)
        joined=previous[['customer_id','persona','event_history_group','is_observed']].merge(
            meta[['customer_id','is_observed','acquisition_pair_valid','has_acquisition_month']],
            on='customer_id',how='left',validate='one_to_one',suffixes=('_source','_next'))
        joined['label_valid']=joined.acquisition_pair_valid.eq(True).fillna(False)
        joined['positive']=joined.has_acquisition_month.eq(True).fillna(False)&joined.label_valid
        for (label,event_group),g in joined.groupby(['persona','event_history_group']):
            evaluation_rows.append(dict(as_of_month=source_month,persona=label,event_history_group=event_group,n=len(g),
                label_n=int(g.label_valid.sum()),positive_n=int(g.positive.sum()),
                present_next_n=int(g.is_observed_next.eq(True).sum())))
        tr=previous[['customer_id','persona']].merge(d[['customer_id','persona']],on='customer_id',validate='one_to_one',suffixes=('_from','_to'))
        t=tr.groupby(['persona_from','persona_to']).size().rename('n').reset_index();t['source_month']=source_month
        transition_rows.append(t)
    previous=d[['customer_id','persona','event_history_group','is_observed']].copy();previous_complete=current_complete
profiles=pd.DataFrame(profile_rows);evaluation=pd.DataFrame(evaluation_rows)
profiles.to_csv(OUT/'persona_profiles.csv',index=False)
pd.DataFrame(event_profile_rows).to_csv(OUT/'event_history_profiles.csv',index=False)
evaluation.to_csv(OUT/'persona_evaluation_all.csv',index=False)
display(profiles.loc[profiles.as_of_month.eq(RECOMMEND_MONTH)])


## 3. Kiểm chứng 1 tháng và độ tin cậy
Acquisition chỉ đo trên cặp đủ dữ liệu; coverage phải báo kèm. Với nhóm vắng snapshot tại t, không đủ điều kiện đo acquisition t→t+1; xem present_next_rate riêng, không điền acquisition bằng 0. Baseline acquisition là toàn cohort có nhãn của cùng tháng. Khoảng Wilson là mô tả từng tháng; khách lặp lại qua tháng nên không coi customer-month là mẫu độc lập để suy luận tổng hợp.

Các tháng 12/2015–04/2016 đã từng được xem trong dự án: đây là **development backtest**, không phải holdout mới. Tỷ lệ ở lại persona là chỉ số mô tả, không tối đa hóa một cách máy móc.

In [ ]:
def wilson(k,n):
    n=np.asarray(n,float);k=np.asarray(k,float)
    lo=np.full_like(k,np.nan);hi=np.full_like(k,np.nan);ok=n>0
    nn=n[ok];pp=k[ok]/nn;z=1.96
    center=(pp+z*z/(2*nn))/(1+z*z/nn)
    half=z*np.sqrt(pp*(1-pp)/nn+z*z/(4*nn*nn))/(1+z*z/nn)
    lo[ok]=center-half;hi[ok]=center+half
    return lo,hi
v=evaluation.loc[evaluation.as_of_month.isin(EVAL_MONTHS)].copy()
assert len(v),'Không có tháng đánh giá'
v['label_coverage']=v.label_n/v.n
v['acquisition_rate']=v.positive_n/v.label_n.replace(0,np.nan)
v['present_next_rate']=v.present_next_n/v.n
base=v.groupby('as_of_month')[['label_n','positive_n']].sum()
base['baseline_rate']=base.positive_n/base.label_n.replace(0,np.nan)
v=v.merge(base[['baseline_rate']],left_on='as_of_month',right_index=True)
v['lift']=v.acquisition_rate/v.baseline_rate.replace(0,np.nan)
v['ci_low'],v['ci_high']=wilson(v.positive_n,v.label_n)
v['weak_evidence']=(v.label_n<MIN_GROUP_N)|(v.positive_n<MIN_POSITIVES)
v.to_csv(OUT/'persona_validation_1m.csv',index=False)
display(v.round(4))
pv=v.groupby(['as_of_month','persona'])[['label_n','positive_n']].sum().reset_index()
pv['acquisition_rate']=pv.positive_n/pv.label_n.replace(0,np.nan)
plot=pv.pivot(index='as_of_month',columns='persona',values='acquisition_rate')
plot.plot(figsize=(15,7),marker='o',title='Acquisition tháng sau trên cặp hợp lệ — development')
plt.ylabel('Tỷ lệ');plt.xticks(rotation=30);plt.tight_layout()
plt.savefig(OUT/'persona_acquisition_rates.png',bbox_inches='tight');plt.show()
transitions=pd.concat(transition_rows,ignore_index=True)
transitions.to_csv(OUT/'persona_transitions_counts.csv',index=False)
last_source=str(pd.Period(RECOMMEND_MONTH,'M')-1)
t=transitions.loc[transitions.source_month.eq(last_source)]
matrix=t.pivot_table(index='persona_from',columns='persona_to',values='n',aggfunc='sum',fill_value=0)
rates=matrix.div(matrix.sum(axis=1).replace(0,np.nan),axis=0)
rates.to_csv(OUT/'persona_transition_rates_latest.csv');display(rates.round(3))
fig,ax=plt.subplots(figsize=(18,13));im=ax.imshow(rates.fillna(0),vmin=0,vmax=1,cmap='Blues')
ax.set_xticks(range(len(rates.columns)),rates.columns,rotation=65,ha='right')
ax.set_yticks(range(len(rates.index)),rates.index);ax.set_title(f'Chuyển persona {last_source} → {RECOMMEND_MONTH}')
fig.colorbar(im,ax=ax);plt.tight_layout();plt.savefig(OUT/'persona_transitions.png',bbox_inches='tight');plt.show()

# So sánh riêng lịch sử event: mỗi dòng thuộc 1 nhóm; không cộng khách hai lần.
ev=v.groupby(['as_of_month','event_history_group'],as_index=False)[['n','label_n','positive_n','present_next_n']].sum()
ev['label_coverage']=ev.label_n/ev.n
ev['acquisition_rate']=ev.positive_n/ev.label_n.replace(0,np.nan)
ev['present_next_rate']=ev.present_next_n/ev.n
ev=ev.merge(base[['baseline_rate']],left_on='as_of_month',right_index=True)
ev['lift']=ev.acquisition_rate/ev.baseline_rate.replace(0,np.nan)
ev['ci_low'],ev['ci_high']=wilson(ev.positive_n,ev.label_n)
ev['weak_evidence']=(ev.label_n<MIN_GROUP_N)|(ev.positive_n<MIN_POSITIVES)
ev.to_csv(OUT/'event_history_validation_1m.csv',index=False)
display(ev.round(4))
event_plot=ev.pivot(index='as_of_month',columns='event_history_group',values='acquisition_rate')
event_plot.plot(figsize=(12,5),marker='o',title='Lịch sử event → acquisition tháng sau')
plt.ylabel('Tỷ lệ trên cặp hợp lệ');plt.xticks(rotation=30);plt.tight_layout()
plt.savefig(OUT/'event_history_acquisition_rates.png',bbox_inches='tight');plt.show()
# Mức khác biệt bên trong nhóm hẹp v1 gần 80%: nhóm nào cần giữ riêng?
latest=pd.read_parquet(OUT/f'personas_{RECOMMEND_MONTH}.parquet')
audit=latest.loc[latest.persona_base.eq('Danh mục hẹp, ít mở gần đây')]
display(pd.crosstab(audit.event_history_group,audit.F4_level,margins=True))
# Ổn định nhóm event, riêng với chuyển persona; nguồn không có tháng t+1 bị loại rõ ràng.
et=[]
for a,b in zip(months,months[1:]):
    if a not in EVAL_MONTHS:continue
    left=pd.read_parquet(OUT/f'personas_{a}.parquet',columns=['customer_id','event_history_group'])
    right=pd.read_parquet(OUT/f'personas_{b}.parquet',columns=['customer_id','event_history_group'])
    joined=left.merge(right,on='customer_id',how='inner',validate='one_to_one',suffixes=('_from','_to'))
    counts=joined.groupby(['event_history_group_from','event_history_group_to']).size().rename('n').reset_index()
    counts['source_month']=a;et.append(counts)
if et:
    event_transitions=pd.concat(et,ignore_index=True)
    event_transitions.to_csv(OUT/'event_history_transitions.csv',index=False)
    tm=event_transitions.pivot_table(index='event_history_group_from',columns='event_history_group_to',values='n',aggfunc='sum',fill_value=0)
    display(tm.div(tm.sum(axis=1),axis=0).round(3))


## 4. Thẻ persona và liên kết CLV / affinity
Persona mô tả chân dung; CLV-proxy ưu tiên khách; affinity xếp sản phẩm chưa sở hữu. Không nhân CLV với ranking_score rồi gọi là doanh thu kỳ vọng. Mã dưới gắn CLV đúng tháng và horizon; để kết nối recommendation của 06, dùng customer_id + as_of_month, kiểm tra mỗi khách có tối đa 7 sản phẩm và không đề xuất sản phẩm đã có. Chưa tự đọc file recommendation vì cần chọn đúng model đã khóa validation của 06.

Đề xuất sản phẩm chỉ là bước tạo ứng viên; chưa có bằng chứng chiến dịch gây ra tăng trưởng nếu chưa A/B test.

In [ ]:
ACTIONS={
'Quan hệ mới':'Hướng dẫn sử dụng và đánh giá nhu cầu sản phẩm nền tảng.',
'Vừa quay lại panel':'Xác minh danh mục hiện tại trước khi đưa đề xuất.',
'Đang mở rộng lặp lại':'Thử bán chéo sản phẩm chưa sở hữu theo affinity; đo MAP@7 và conversion.',
'Danh mục rộng, ít mở gần đây':'Duy trì danh mục; kiểm tra số sản phẩm còn có thể giới thiệu.',
'Danh mục hẹp, ít mở gần đây':'Tìm nhu cầu phù hợp; không mặc định khách không hoạt động.',
'Vắng snapshot ngắn':'Kiểm tra nguyên nhân vắng dữ liệu; chưa suy ra đã rời ngân hàng.',
'Vắng snapshot dài':'Theo dõi tỷ lệ quay lại panel; chưa xếp sản phẩm từ danh mục cũ.',
'Lịch sử quan sát ngắn':'Thu thập thêm lịch sử; dùng baseline phù hợp.',
'Chưa đủ thông tin':'Kiểm tra chất lượng snapshot.',
'Thiếu lịch sử acquisition':'Kiểm tra tính nhất quán R/F upstream.',
'Danh mục khác':'Tiếp tục mô tả; chỉ tách nhóm khi có bằng chứng khác biệt.'}
current=pd.read_parquet(OUT/f'personas_{RECOMMEND_MONTH}.parquet')
clv_status='not_provided'
if CLV_PATH:
    c=pd.read_parquet(CLV_PATH)
    assert c.customer_id.is_unique and c.customer_id.notna().all()
    assert c.as_of_month.astype(str).eq(RECOMMEND_MONTH).all()
    assert c.horizon_months.eq(1).all(),'Chỉ ghép CLV 1 tháng trong bản này'
    assert c.predicted_points.notna().all() and np.isfinite(c.predicted_points).all()
    current=current.merge(c[['customer_id','predicted_points']],on='customer_id',how='left',validate='one_to_one')
    clv_status='joined_1m'
    clv=current.groupby('persona').agg(clv_n=('predicted_points','count'),clv_points_median=('predicted_points','median'))
    clv.to_csv(OUT/'persona_clv_summary.csv')
current.to_parquet(OUT/f'persona_activation_{RECOMMEND_MONTH}.parquet',index=False)
cards=[]
for _,r in profiles.loc[profiles.as_of_month.eq(RECOMMEND_MONTH)].iterrows():
    cards.append(f"## {r.persona}\n- Quy mô: {int(r.n):,} ({r.share:.1%}).\n- Trung vị: R={r.R_median:.2f}, F={r.F_median:.2f}, số sản phẩm={r.products_median:.2f}.\n- Hành động đề xuất: {ACTIONS.get(r.persona, 'Đối chiếu acquisition/lift và coverage; chọn khách bằng CLV-proxy, sản phẩm bằng affinity chưa sở hữu; chưa suy ra không sử dụng ngân hàng.')}\n- Bằng chứng tương lai: xem persona_validation_1m.csv, gồm coverage và khoảng tin cậy.\n")
(OUT/'persona_cards.md').write_text('# Persona — '+RECOMMEND_MONTH+'\n\n'+'\n'.join(cards),encoding='utf-8')
config={k:globals()[k] for k in ['RULE_VERSION','MIN_VALID_PAIRS','RECENT_R','FREQUENT_F','WIDE_PRODUCTS','NARROW_PRODUCTS','EVAL_MONTHS','RECOMMEND_MONTH']}
config.update(feature_dir=str(feature_dir),lifecycle_dir=str(life_dir),clv_status=clv_status,
    evaluation_scope='development backtest; 1m complete-pair acquisition; panel presence separate',
    source_notebooks=['01','05'],optional_sources=['07'],event_windows=[3,6],event_source='05 acquisition_pair_valid + has_acquisition_month',F4='1 / 2 / 3-4 / >=5; zero separate')
(OUT/'persona_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Đã xuất:',OUT)


## 5. Viết kết luận sau Run All
1. Nhóm nào lớn nhất? Nhóm khác/thiếu thông tin có quá lớn không?
2. Các nhóm có khác nhau về danh mục và acquisition tháng sau qua nhiều tháng không? Đọc coverage trước lift.
3. Nhóm nào có ít event, CI rộng, chưa đủ bằng chứng để ưu tiên?
4. Khách đổi persona vì acquisition, thay đổi danh mục hay lifecycle? Không lấy tỷ lệ ở lại cao làm mục tiêu duy nhất.
5. M_group_points có khác biệt trong từng persona không? Nếu khác biệt lớn, dùng làm tag ưu tiên thay vì thêm hàng chục persona.
6. Nếu có CLV: độ phủ và trung vị CLV-proxy theo persona; không gọi là tiền.
7. Bước tiếp: ghép recommendation 06 theo đúng model khóa; thử A/B với nhóm đối chứng ngẫu nhiên, đo conversion incremental và đóng sản phẩm. Cần thiết kế sample size theo baseline thực tế.

**Chưa bao gồm:** hồ sơ tuổi/thu nhập do không có trong output 01/05 dùng ở đây; tự động ghép affinity; bằng chứng campaign causal; xác nhận churn ngân hàng. Có thể bổ sung hồ sơ as-of sau, nhưng phải báo missing và không dùng metadata tương lai.

8. Trong cùng danh mục hẹp/rộng: NoObservedEvent, không event 3 tháng và không event 6 tháng khác nhau bao nhiêu? Đọc event_history_validation_1m.csv.
9. NoEvent3m_Unknown6m và InsufficientRecentObservation là nhóm thiếu coverage, không diễn giải là hành vi xấu.
10. Có quá nhiều tổ hợp ít mẫu không? Chỉ gộp sau khi xem đặc điểm và độ tin cậy; báo rõ đó là điều chỉnh trên development.
